<a href="https://colab.research.google.com/github/will-mccormack/CS-M148-Proj/blob/main/COM_SCI_M148_NN_with_genre.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from skimage import io, transform
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, utils
from sklearn.preprocessing import StandardScaler
import time
from sklearn.metrics import r2_score, mean_squared_error

import warnings
warnings.filterwarnings("ignore")

In [2]:
train_filepath = "https://raw.githubusercontent.com/will-mccormack/CS-M148-Proj/main/Data/train.csv"
validation_filepath = "https://raw.githubusercontent.com/will-mccormack/CS-M148-Proj/main/Data/validation.csv"

#train_data = pd.read_csv(train_filepath)
#validation_data = pd.read_csv(train_filepath)

In [3]:
# device setup to find gpu
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [13]:
def load_data(train_path, val_path):
  train_data = pd.read_csv(train_path)
  val_data = pd.read_csv(val_path)
  train_data['explicit'] = train_data['explicit'].astype(int)
  val_data['explicit'] = val_data['explicit'].astype(int)
  train_data = pd.get_dummies(train_data, columns=['track_genre'], dtype=float)
  val_data = pd.get_dummies(val_data, columns=['track_genre'], dtype=float)
  train_cols =train_data.columns
  val_data = val_data.reindex(columns=train_cols, fill_value=0)
  return train_data, val_data


In [14]:
def calculate_metrics(model, loader, device):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for features, labels in loader:
            features = features.to(device)
            preds = model(features).cpu().numpy() # Move to CPU for sklearn

            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    # Convert lists to arrays
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    # Calculate Metrics
    mse = mean_squared_error(all_labels, all_preds)
    rmse = np.sqrt(mse)
    r2 = r2_score(all_labels, all_preds)

    return mse, rmse, r2

In [5]:
class SpotifyDataset(Dataset):
  def __init__(self, dataframe,scaler=None,is_train=True):
    # split x and y
    self.x = dataframe.drop(columns=['popularity','instrumentalness','time_signature','valence']).values
    self.y = dataframe['popularity'].values
    if is_train:
      self.scaler = StandardScaler()
      self.x = self.scaler.fit_transform(self.x) #scale data
    else:
      self.scaler = scaler
      self.x = self.scaler.transform(self.x)

  def __len__(self):
    return len(self.x)

  def __getitem__(self, idx):
    features = self.x[idx]
    label = self.y[idx]
    features_tensor = torch.tensor(features, dtype=torch.float32)
    label_tensor = torch.tensor(label, dtype=torch.float32)
    return features_tensor, label_tensor

In [6]:
train_data, val_data = load_data(train_filepath, validation_filepath)
train_dataset = SpotifyDataset(train_data, is_train=True)
val_dataset = SpotifyDataset(val_data, scaler=train_dataset.scaler, is_train=False)

In [7]:
BATCH_SIZE = 1024
train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2,pin_memory=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2,pin_memory=True)

input_size = train_dataset.x.shape[1]
print(f"Input size: {input_size}")

Input size: 124


# NN

## Hyperparameter Tuning

In [15]:
learning_rates = [0.01, 0.001, 0.0001]
results = {}

for lr in learning_rates:
    print(f"\nTesting Learning Rate: {lr}")

    # 1. Re-initialize model (Fresh start for every LR)
    # Note: Use input_size_pca if using PCA data, or dataset.x.shape[1] if using raw data
    input_dim = train_dataset.x.shape[1]
    temp_model = nn.Sequential(
        nn.Linear(input_dim, 64),
        nn.ReLU(),
        nn.Linear(64, 32),
        nn.ReLU(),
        nn.Linear(32, 1)
    ).to(device)

    optimizer = optim.Adam(temp_model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    # 2. Short Training Run (e.g., 5-10 epochs is enough to see the trend)
    for epoch in range(10):
        temp_model.train()
        for features, labels in train_loader: # Use your PCA loader here
            features, labels = features.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = temp_model(features)
            loss = criterion(outputs, labels.view(-1, 1))
            loss.backward()
            optimizer.step()

    # 3. Check Validation Loss
    mse, rmse, r2 = calculate_metrics(temp_model, val_loader, device)
    results[lr] = rmse
    print(f"LR {lr} -> Validation RMSE: {rmse:.4f}")

# Find best LR
best_lr = min(results, key=results.get)
print(f"\nBest Learning Rate found: {best_lr}")


Testing Learning Rate: 0.01
LR 0.01 -> Validation RMSE: 19.1879

Testing Learning Rate: 0.001
LR 0.001 -> Validation RMSE: 19.3345

Testing Learning Rate: 0.0001
LR 0.0001 -> Validation RMSE: 31.7959

Best Learning Rate found: 0.01


## NN Training

In [17]:
model = nn.Sequential(
    # Layer 1
    nn.Linear(input_size,128),
    nn.BatchNorm1d(128),
    nn.ReLU(),
    nn.Dropout(0.2),
    # Layer 2
    nn.Linear(128,64),
    nn.BatchNorm1d(64),
    nn.ReLU(),
    nn.Dropout(0.1),
    # Layer 3
    nn.Linear(64,32),
    nn.BatchNorm1d(32),
    nn.ReLU(),
    # Layer 4
    nn.Linear(32,16),
    nn.ReLU(),
    # Layer 5
    nn.Linear(16,1)
)

model = model.to(device) # move to GPU

loss_type = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=best_lr)

num_epochs = 100
patience = 10
best_val_loss = float('inf')
epochs_no_improve = 0

print("Starting Training")

start_time = time.time()

for epoch in range(num_epochs):
  model.train()
  epoch_loss = 0.0

  for features, labels in train_loader:
    features, labels = features.to(device), labels.to(device) # move data to GPU

    optimizer.zero_grad()
    # forward pass, predicted
    predictions = model(features)
    # loss
    loss = loss_type(predictions, labels.view(-1,1))
    #backprop, gradients and update weights

    loss.backward()
    optimizer.step()
    #update loss
    epoch_loss += loss.item()

  # validation test
  model.eval()
  val_loss = 0.0
  with torch.no_grad():
    for features, labels in val_loader:
      features, labels = features.to(device), labels.to(device)
      predictions = model(features)
      val_loss += loss_type(predictions, labels.view(-1,1)).item()

  avg_train_loss = epoch_loss / len(train_loader)
  avg_val_loss = val_loss / len(val_loader)

  if (epoch + 1) % 5 == 0:
    print(f"Epoch {epoch+1}/{num_epochs} | Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f}")
  if avg_val_loss < best_val_loss:
    best_val_loss = avg_val_loss
    patience_counter = 0
  else:
    patience_counter += 1
    if patience_counter >= patience:
      print(f"Early stopping at epoch {epoch+1}")
      break

print("Training finished")
print(f"Done in {time.time() - start_time:.2f}s")

Starting Training
Epoch 5/100 | Train: 359.6872 | Val: 368.0004
Epoch 10/100 | Train: 348.8399 | Val: 360.4107
Epoch 15/100 | Train: 341.6234 | Val: 354.9604
Epoch 20/100 | Train: 336.5386 | Val: 355.8853
Epoch 25/100 | Train: 330.9675 | Val: 355.0958
Epoch 30/100 | Train: 325.6445 | Val: 352.8135
Epoch 35/100 | Train: 320.1929 | Val: 350.0208
Epoch 40/100 | Train: 316.1293 | Val: 351.2132
Epoch 45/100 | Train: 312.4177 | Val: 347.4831
Epoch 50/100 | Train: 309.6923 | Val: 353.2505
Epoch 55/100 | Train: 306.0628 | Val: 345.4452
Epoch 60/100 | Train: 302.4600 | Val: 348.6683
Early stopping at epoch 63
Training finished
Done in 202.99s


In [18]:
mse, rmse, r2 = calculate_metrics(model, val_loader, device)
print(f"Final Validation Results -> MSE: {mse:.2f} | RMSE: {rmse:.2f} | R2: {r2:.4f}")

Final Validation Results -> MSE: 345.72 | RMSE: 18.59 | R2: 0.3044
